# Extended MD Analyses — AI-Designed Protein Engineering WorkflowThese cells extend notebook 07/08 with 10 additional analyses that close thecomputational biology design loop:| # | Analysis | What it tells you ||---|----------|-------------------|| 15 | Interface RMSD | Is the binder drifting off the target? || 16 | Hydrogen bond validation | Which polar contacts are geometrically real? || 17 | Salt bridge & cation-π geometry | Are electrostatic anchors actually formed? || 18 | MM-GBSA binding energy estimation | Approximate binding ΔG per design || 19 | Per-residue energy decomposition | Which residues drive binding energy? || 20 | Dynamic cross-correlation (DCCM) | Do binder and target move together? || 21 | Interface PCA | Dominant collective motions at the interface || 22 | Thermal stability screen | Compare 300K vs 350K contact survival || 23 | Steered MD / COM pulling | Mechanical unbinding resistance || 24 | Hotspot recapitulation scoring | Did RFdiffusion hotspots survive MD? |**Prerequisites:** Run all cells through Cell 14 (pair persistence) first.These cells expect `prepared_systems`, config variables, and analysis outputsto be available in memory or on Drive.

## Cell 15 — Interface RMSD (binder position relative to target)Superpose each frame on **target** backbone Cα, then measure binder Cα RMSD*without* further alignment. This separates two failure modes:- **Self-RMSD** (Cell 10): binder unfolds internally- **Interface RMSD** (this cell): binder detaches or rocks relative to targetA design with low self-RMSD but high interface RMSD is folded but not bound.

In [ ]:
# ============================================================
# Cell 15 — Interface RMSD: binder position relative to target
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json


def compute_interface_rmsd(system_info, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])

    traj_dcd = Path(system_info["traj_dcd"])
    top_pdb = Path(system_info["equilibrated_pdb"])
    out_csv = drive_dir / "interface_rmsd.csv"
    out_png = drive_dir / "interface_rmsd.png"
    out_json = drive_dir / "interface_rmsd_summary.json"

    if out_json.exists() and not force:
        print(f"✓ Interface RMSD skipped for {design_name}; found {out_json}")
        with open(out_json) as f:
            system_info["interface_rmsd"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"Interface RMSD: {design_name}")
    print(f"{'=' * 60}")

    traj = md.load(str(traj_dcd), top=str(top_pdb))
    protein_atoms = traj.topology.select("protein")
    traj_protein = traj.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    # Superpose on TARGET Cα atoms
    target_ca = traj_protein.topology.select(f"chainid {target_chain} and name CA")
    traj_protein.superpose(traj_protein, frame=0, atom_indices=target_ca)

    # Measure binder Cα RMSD WITHOUT further superposition
    binder_ca = traj_protein.topology.select(f"chainid {binder_chain} and name CA")
    ref_xyz = traj_protein.xyz[0, binder_ca, :]
    interface_rmsd = np.sqrt(((traj_protein.xyz[:, binder_ca, :] - ref_xyz) ** 2).sum(axis=2).mean(axis=1)) * 10.0

    # Also get self-RMSD for comparison
    binder_bb = traj_protein.topology.select(f"chainid {binder_chain} and backbone")
    traj_binder_only = traj_protein.atom_slice(binder_bb)
    self_rmsd = md.rmsd(traj_binder_only, traj_binder_only, frame=0) * 10.0

    prod_ns = float(system_info.get("production_ns", PRODUCTION_NS))
    t_ns = np.linspace(0, prod_ns, traj_protein.n_frames)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t_ns, interface_rmsd, lw=0.8, label="Interface RMSD (target-aligned)", color="#E8927C")
    ax.plot(t_ns, self_rmsd, lw=0.8, label="Self RMSD (binder-aligned)", color="#8DB85B")
    ax.axhline(2.0, color="gray", ls="--", alpha=0.5, label="2 Å")
    ax.axhline(5.0, color="red", ls="--", alpha=0.3, label="5 Å (drift threshold)")
    ax.set(xlabel="Time (ns)", ylabel="RMSD (Å)",
           title=f"Interface vs Self RMSD — {design_name}")
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.show()

    results = {
        "interface_rmsd_mean": float(np.mean(interface_rmsd)),
        "interface_rmsd_final": float(interface_rmsd[-1]),
        "interface_rmsd_max": float(np.max(interface_rmsd)),
        "self_rmsd_mean": float(np.mean(self_rmsd)),
        "self_rmsd_final": float(self_rmsd[-1]),
        "ratio_interface_to_self": float(np.mean(interface_rmsd) / max(np.mean(self_rmsd), 0.01)),
    }

    pd.DataFrame({"time_ns": t_ns, "interface_rmsd_A": interface_rmsd, "self_rmsd_A": self_rmsd}).to_csv(out_csv, index=False)
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["interface_rmsd"] = results
    print(f"  Interface RMSD: mean={results['interface_rmsd_mean']:.2f} Å, final={results['interface_rmsd_final']:.2f} Å")
    print(f"  Self RMSD:      mean={results['self_rmsd_mean']:.2f} Å")
    print(f"  Ratio (interface/self): {results['ratio_interface_to_self']:.2f}")
    print(f"  ✓ {out_png}")
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("traj_dcd") and Path(s["traj_dcd"]).exists():
        prepared_systems[i] = compute_interface_rmsd(s)
save_manifest(prepared_systems)

## Cell 16 — Hydrogen bond analysis with geometric validationIdentifies **actual** hydrogen bonds at the binder–target interface usingthe Baker–Hubbard criterion (D-H···A angle > 120°, H···A distance < 2.5 Å).Upgrades the `candidate_polar_hbond` labels from Cell 14 to geometricallyconfirmed H-bonds with per-frame occupancies.

In [ ]:
# ============================================================
# Cell 16 — Interface hydrogen bond analysis (Baker-Hubbard)
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json


def analyze_interface_hbonds(system_info, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])

    traj_dcd = Path(system_info["traj_dcd"])
    top_pdb = Path(system_info["equilibrated_pdb"])
    out_csv = drive_dir / "interface_hbonds.csv"
    out_png = drive_dir / "interface_hbonds.png"
    out_json = drive_dir / "interface_hbonds_summary.json"

    if out_json.exists() and not force:
        print(f"✓ H-bond analysis skipped for {design_name}")
        with open(out_json) as f:
            system_info["hbond_analysis"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"H-bond analysis: {design_name}")
    print(f"{'=' * 60}")

    traj = md.load(str(traj_dcd), top=str(top_pdb))
    protein_atoms = traj.topology.select("protein")
    traj_protein = traj.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    binder_residue_indices = {r.index for r in traj_protein.topology.residues if r.chain.index == binder_chain}
    target_residue_indices = {r.index for r in traj_protein.topology.residues if r.chain.index == target_chain}

    # Baker-Hubbard: D-H···A angle > 120°, H···A < 2.5 Å
    # Returns array of (donor_idx, hydrogen_idx, acceptor_idx) per frame
    hbonds_by_frame = md.baker_hubbard(traj_protein, freq=0.0)

    # For each frame, find interface H-bonds
    # baker_hubbard returns bonds present in >= freq fraction of frames
    # With freq=0.0 we get all ever-observed bonds
    # Now compute per-frame presence
    n_frames = traj_protein.n_frames

    # Get all unique hbond triplets
    if len(hbonds_by_frame) == 0:
        print("  No hydrogen bonds detected")
        results = {"n_interface_hbonds": 0, "interface_hbond_occupancy_mean": 0.0}
        with open(out_json, "w") as f:
            json.dump(results, f, indent=2)
        system_info["hbond_analysis"] = results
        return system_info

    # Filter for interface H-bonds: donor in one chain, acceptor in the other
    interface_hbonds = []
    for donor_idx, h_idx, acceptor_idx in hbonds_by_frame:
        donor_res = traj_protein.topology.atom(donor_idx).residue.index
        acceptor_res = traj_protein.topology.atom(acceptor_idx).residue.index

        is_interface = (
            (donor_res in binder_residue_indices and acceptor_res in target_residue_indices) or
            (donor_res in target_residue_indices and acceptor_res in binder_residue_indices)
        )
        if is_interface:
            interface_hbonds.append((donor_idx, h_idx, acceptor_idx))

    print(f"  Total H-bond triplets ever observed: {len(hbonds_by_frame)}")
    print(f"  Interface H-bond triplets: {len(interface_hbonds)}")

    if len(interface_hbonds) == 0:
        results = {"n_interface_hbonds": 0, "interface_hbond_occupancy_mean": 0.0}
        with open(out_json, "w") as f:
            json.dump(results, f, indent=2)
        system_info["hbond_analysis"] = results
        return system_info

    # Compute per-frame occupancy for each interface H-bond
    # Use distance + angle criteria frame by frame
    rows = []
    hbond_timeseries = []

    for donor_idx, h_idx, acceptor_idx in interface_hbonds:
        # H···A distance
        ha_dist = md.compute_distances(traj_protein, [[h_idx, acceptor_idx]]).flatten()
        # D-H···A angle
        dha_angle = md.compute_angles(traj_protein, [[donor_idx, h_idx, acceptor_idx]]).flatten()

        present = (ha_dist < 0.25) & (dha_angle > (120.0 * np.pi / 180.0))
        occupancy = float(np.mean(present))

        donor_atom = traj_protein.topology.atom(donor_idx)
        acceptor_atom = traj_protein.topology.atom(acceptor_idx)
        donor_res = donor_atom.residue
        acceptor_res = acceptor_atom.residue

        # Determine direction
        if donor_res.index in binder_residue_indices:
            binder_res, target_res = donor_res, acceptor_res
            direction = "binder_donor"
        else:
            binder_res, target_res = acceptor_res, donor_res
            direction = "target_donor"

        rows.append({
            "design_name": design_name,
            "binder_residue": f"{binder_res.name}{binder_res.resSeq}",
            "target_residue": f"{target_res.name}{target_res.resSeq}",
            "donor_atom": f"{donor_atom.residue.name}{donor_atom.residue.resSeq}.{donor_atom.name}",
            "acceptor_atom": f"{acceptor_atom.residue.name}{acceptor_atom.residue.resSeq}.{acceptor_atom.name}",
            "direction": direction,
            "occupancy": occupancy,
            "mean_ha_distance_A": float(np.mean(ha_dist) * 10.0),
            "mean_dha_angle_deg": float(np.mean(dha_angle) * 180.0 / np.pi),
        })
        hbond_timeseries.append(present.astype(int))

    df_hbonds = pd.DataFrame(rows).sort_values("occupancy", ascending=False).reset_index(drop=True)
    df_hbonds.to_csv(out_csv, index=False)

    # Plot: total interface H-bonds over time
    hbond_matrix = np.array(hbond_timeseries)  # (n_hbonds, n_frames)
    total_hbonds_per_frame = hbond_matrix.sum(axis=0)

    prod_ns = float(system_info.get("production_ns", PRODUCTION_NS))
    t_ns = np.linspace(0, prod_ns, n_frames)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Interface H-bonds — {design_name}", fontweight="bold")

    axes[0].plot(t_ns, total_hbonds_per_frame, lw=0.8, color="#4A90D9")
    axes[0].set(xlabel="Time (ns)", ylabel="# interface H-bonds", title="H-bond count over time")

    # Bar chart of top H-bonds by occupancy
    top_n = min(15, len(df_hbonds))
    df_top = df_hbonds.head(top_n)
    labels = [f"{r['binder_residue']}↔{r['target_residue']}" for _, r in df_top.iterrows()]
    axes[1].barh(range(top_n), df_top["occupancy"].values, color="#E8927C")
    axes[1].set_yticks(range(top_n))
    axes[1].set_yticklabels(labels, fontsize=7)
    axes[1].set(xlabel="Occupancy", title=f"Top {top_n} interface H-bonds")
    axes[1].invert_yaxis()

    plt.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.show()

    results = {
        "n_interface_hbonds": len(interface_hbonds),
        "n_persistent_hbonds_gt50pct": int((df_hbonds["occupancy"] > 0.5).sum()),
        "n_persistent_hbonds_gt80pct": int((df_hbonds["occupancy"] > 0.8).sum()),
        "interface_hbond_occupancy_mean": float(df_hbonds["occupancy"].mean()),
        "total_hbonds_mean_per_frame": float(np.mean(total_hbonds_per_frame)),
    }

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["hbond_analysis"] = results
    print(f"  Persistent H-bonds (>50% occ): {results['n_persistent_hbonds_gt50pct']}")
    print(f"  Persistent H-bonds (>80% occ): {results['n_persistent_hbonds_gt80pct']}")
    print(f"  ✓ {out_csv}")

    display(df_hbonds.head(15))
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("traj_dcd") and Path(s["traj_dcd"]).exists():
        prepared_systems[i] = analyze_interface_hbonds(s)
save_manifest(prepared_systems)

## Cell 17 — Salt bridge and cation-π geometric validationValidates electrostatic anchors at the interface:- **Salt bridges**: distance between charge-center atoms (NZ/NH1/NH2 for LYS/ARG,  OD1/OD2/OE1/OE2 for ASP/GLU) < 4.0 Å- **Cation-π**: distance from cation (NZ/NH1/NH2) to aromatic ring centroid < 6.0 ÅUpgrades `candidate_salt_bridge` and `candidate_cation_pi` from Cell 14.

In [ ]:
# ============================================================
# Cell 17 — Salt bridge and cation-π geometric validation
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json

SALT_BRIDGE_CUTOFF_NM = 0.40   # 4.0 Å between charge centers
CATION_PI_CUTOFF_NM = 0.60     # 6.0 Å cation to ring centroid

CATION_ATOMS = {
    "LYS": ["NZ"],
    "ARG": ["NH1", "NH2", "NE"],
}
ANION_ATOMS = {
    "ASP": ["OD1", "OD2"],
    "GLU": ["OE1", "OE2"],
}
AROMATIC_RING_ATOMS = {
    "PHE": ["CG", "CD1", "CD2", "CE1", "CE2", "CZ"],
    "TYR": ["CG", "CD1", "CD2", "CE1", "CE2", "CZ"],
    "TRP": ["CG", "CD1", "CD2", "NE1", "CE2", "CE3", "CZ2", "CZ3", "CH2"],
    "HIS": ["CG", "ND1", "CD2", "CE1", "NE2"],
}


def get_atom_indices(topology, residue, atom_names):
    """Get atom indices for named atoms in a residue."""
    indices = []
    for atom in residue.atoms:
        if atom.name in atom_names:
            indices.append(atom.index)
    return indices


def analyze_electrostatic_contacts(system_info, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    out_csv = drive_dir / "electrostatic_contacts.csv"
    out_png = drive_dir / "electrostatic_contacts.png"
    out_json = drive_dir / "electrostatic_contacts_summary.json"

    if out_json.exists() and not force:
        print(f"✓ Electrostatic analysis skipped for {design_name}")
        with open(out_json) as f:
            system_info["electrostatic_analysis"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"Electrostatic contacts: {design_name}")
    print(f"{'=' * 60}")

    traj = md.load(str(system_info["traj_dcd"]), top=str(system_info["equilibrated_pdb"]))
    protein_atoms = traj.topology.select("protein")
    traj_protein = traj.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    binder_residues = [r for r in traj_protein.topology.residues if r.chain.index == binder_chain]
    target_residues = [r for r in traj_protein.topology.residues if r.chain.index == target_chain]

    n_frames = traj_protein.n_frames
    prod_ns = float(system_info.get("production_ns", PRODUCTION_NS))
    t_ns = np.linspace(0, prod_ns, n_frames)

    rows = []

    # --- Salt bridges ---
    for b_res in binder_residues:
        for t_res in target_residues:
            # Check both directions: binder-cation/target-anion and vice versa
            pairs_to_check = []

            if b_res.name in CATION_ATOMS and t_res.name in ANION_ATOMS:
                cat_idx = get_atom_indices(traj_protein.topology, b_res, CATION_ATOMS[b_res.name])
                ani_idx = get_atom_indices(traj_protein.topology, t_res, ANION_ATOMS[t_res.name])
                pairs_to_check.append((cat_idx, ani_idx, "binder_cation"))

            if t_res.name in CATION_ATOMS and b_res.name in ANION_ATOMS:
                cat_idx = get_atom_indices(traj_protein.topology, t_res, CATION_ATOMS[t_res.name])
                ani_idx = get_atom_indices(traj_protein.topology, b_res, ANION_ATOMS[b_res.name])
                pairs_to_check.append((cat_idx, ani_idx, "target_cation"))

            for cat_indices, ani_indices, direction in pairs_to_check:
                if not cat_indices or not ani_indices:
                    continue
                # Minimum distance across all cation-anion atom pairs per frame
                atom_pairs = np.array([(c, a) for c in cat_indices for a in ani_indices])
                dists = md.compute_distances(traj_protein, atom_pairs)
                min_dist = dists.min(axis=1)
                present = min_dist < SALT_BRIDGE_CUTOFF_NM
                occupancy = float(np.mean(present))

                rows.append({
                    "design_name": design_name,
                    "interaction_type": "salt_bridge",
                    "binder_residue": f"{b_res.name}{b_res.resSeq}",
                    "target_residue": f"{t_res.name}{t_res.resSeq}",
                    "direction": direction,
                    "occupancy": occupancy,
                    "mean_distance_A": float(np.mean(min_dist) * 10.0),
                    "min_distance_A": float(np.min(min_dist) * 10.0),
                })

    # --- Cation-π ---
    for b_res in binder_residues:
        for t_res in target_residues:
            checks = []

            if b_res.name in CATION_ATOMS and t_res.name in AROMATIC_RING_ATOMS:
                checks.append((b_res, t_res, "binder_cation"))
            if t_res.name in CATION_ATOMS and b_res.name in AROMATIC_RING_ATOMS:
                checks.append((t_res, b_res, "target_cation"))

            for cat_res, arom_res, direction in checks:
                cat_indices = get_atom_indices(traj_protein.topology, cat_res, CATION_ATOMS[cat_res.name])
                ring_indices = get_atom_indices(traj_protein.topology, arom_res, AROMATIC_RING_ATOMS[arom_res.name])

                if not cat_indices or not ring_indices:
                    continue

                # Ring centroid
                ring_xyz = traj_protein.xyz[:, ring_indices, :]  # (frames, n_ring_atoms, 3)
                centroid = ring_xyz.mean(axis=1)  # (frames, 3)

                # Min distance from any cation atom to centroid
                min_dists = np.full(n_frames, np.inf)
                for ci in cat_indices:
                    cat_xyz = traj_protein.xyz[:, ci, :]  # (frames, 3)
                    d = np.linalg.norm(cat_xyz - centroid, axis=1)
                    min_dists = np.minimum(min_dists, d)

                present = min_dists < CATION_PI_CUTOFF_NM
                occupancy = float(np.mean(present))

                rows.append({
                    "design_name": design_name,
                    "interaction_type": "cation_pi",
                    "binder_residue": f"{b_res.name}{b_res.resSeq}" if b_res.chain.index == binder_chain else f"{arom_res.name}{arom_res.resSeq}",
                    "target_residue": f"{t_res.name}{t_res.resSeq}" if t_res.chain.index == target_chain else f"{cat_res.name}{cat_res.resSeq}",
                    "direction": direction,
                    "occupancy": occupancy,
                    "mean_distance_A": float(np.mean(min_dists) * 10.0),
                    "min_distance_A": float(np.min(min_dists) * 10.0),
                })

    df_elec = pd.DataFrame(rows)
    df_elec = df_elec.sort_values("occupancy", ascending=False).reset_index(drop=True)
    df_elec.to_csv(out_csv, index=False)

    df_present = df_elec[df_elec["occupancy"] > 0.0]

    n_salt = int((df_present["interaction_type"] == "salt_bridge").sum())
    n_catpi = int((df_present["interaction_type"] == "cation_pi").sum())
    n_salt_persistent = int(((df_present["interaction_type"] == "salt_bridge") & (df_present["occupancy"] > 0.5)).sum())
    n_catpi_persistent = int(((df_present["interaction_type"] == "cation_pi") & (df_present["occupancy"] > 0.5)).sum())

    results = {
        "n_salt_bridges": n_salt,
        "n_salt_bridges_persistent": n_salt_persistent,
        "n_cation_pi": n_catpi,
        "n_cation_pi_persistent": n_catpi_persistent,
    }

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["electrostatic_analysis"] = results

    print(f"  Salt bridges (any occ): {n_salt}, persistent (>50%): {n_salt_persistent}")
    print(f"  Cation-π (any occ): {n_catpi}, persistent (>50%): {n_catpi_persistent}")
    print(f"  ✓ {out_csv}")

    if len(df_present) > 0:
        fig, ax = plt.subplots(figsize=(10, max(3, len(df_present.head(20)) * 0.35)))
        df_plot = df_present.head(20)
        colors = ["#4A90D9" if t == "salt_bridge" else "#E8927C" for t in df_plot["interaction_type"]]
        labels = [f"{r['binder_residue']}↔{r['target_residue']} ({r['interaction_type']})" for _, r in df_plot.iterrows()]
        ax.barh(range(len(df_plot)), df_plot["occupancy"].values, color=colors)
        ax.set_yticks(range(len(df_plot)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set(xlabel="Occupancy", title=f"Electrostatic contacts — {design_name}")
        ax.invert_yaxis()
        plt.tight_layout()
        fig.savefig(out_png, dpi=200)
        plt.show()

    display(df_present.head(20))
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("traj_dcd") and Path(s["traj_dcd"]).exists():
        prepared_systems[i] = analyze_electrostatic_contacts(s)
save_manifest(prepared_systems)

## Cell 18 — MM-GBSA binding energy estimationApproximates binding free energy using single-trajectory MM-GBSA:```ΔG_bind ≈ E(complex) − E(target) − E(binder)```Each term uses the **same trajectory frames** (single-trajectory approximation)with OpenMM's implicit solvent (OBC2 generalized Born model). This is a roughranking tool, not an absolute ΔG — but it's fast and discriminates good bindersfrom bad ones.**Note:** Requires frames extracted from the explicit-solvent trajectory,stripped of water/ions, and re-evaluated with implicit solvent.

In [ ]:
# ============================================================
# Cell 18 — MM-GBSA binding energy estimation (single trajectory)
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json

import openmm
from openmm import unit
from openmm.app import ForceField, Modeller, NoCutoff, HBonds, PDBFile

# Number of evenly-spaced frames to evaluate (more = better average, slower)
MMGBSA_N_FRAMES = 20


def compute_mmgbsa(system_info, n_frames=MMGBSA_N_FRAMES, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    out_csv = drive_dir / "mmgbsa_energies.csv"
    out_json = drive_dir / "mmgbsa_summary.json"
    out_png = drive_dir / "mmgbsa_energies.png"

    if out_json.exists() and not force:
        print(f"✓ MM-GBSA skipped for {design_name}")
        with open(out_json) as f:
            system_info["mmgbsa"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"MM-GBSA: {design_name}")
    print(f"{'=' * 60}")

    traj = md.load(str(system_info["traj_dcd"]), top=str(system_info["equilibrated_pdb"]))
    protein_atoms = traj.topology.select("protein")
    traj_protein = traj.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    binder_atoms = traj_protein.topology.select(f"chainid {binder_chain}")
    target_atoms = traj_protein.topology.select(f"chainid {target_chain}")

    # Select evenly spaced frames from the second half of the trajectory
    half = traj_protein.n_frames // 2
    frame_indices = np.linspace(half, traj_protein.n_frames - 1, n_frames, dtype=int)

    # Use implicit solvent forcefield (GBSA-OBC2)
    ff = ForceField("amber14-all.xml", "implicit/gbn2.xml")

    def evaluate_energy(topology, positions):
        """Single-point energy with implicit solvent."""
        system = ff.createSystem(topology, nonbondedMethod=NoCutoff, constraints=HBonds)
        integrator = openmm.LangevinMiddleIntegrator(
            300 * unit.kelvin, 1.0 / unit.picoseconds, 0.002 * unit.picoseconds
        )
        context = openmm.Context(system, integrator)
        context.setPositions(positions)
        state = context.getState(getEnergy=True)
        energy_kj = state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
        del context, integrator
        return energy_kj

    # Save a temporary PDB for topology construction
    tmp_dir = Path(system_info.get("work_dir", drive_dir))
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # Write sub-topologies once
    traj_binder = traj_protein.atom_slice(binder_atoms)
    traj_target = traj_protein.atom_slice(target_atoms)

    tmp_complex = tmp_dir / "mmgbsa_complex.pdb"
    tmp_binder = tmp_dir / "mmgbsa_binder.pdb"
    tmp_target = tmp_dir / "mmgbsa_target.pdb"

    traj_protein[0].save_pdb(str(tmp_complex))
    traj_binder[0].save_pdb(str(tmp_binder))
    traj_target[0].save_pdb(str(tmp_target))

    pdb_complex = PDBFile(str(tmp_complex))
    pdb_binder = PDBFile(str(tmp_binder))
    pdb_target = PDBFile(str(tmp_target))

    rows = []
    for fi, frame_idx in enumerate(frame_indices):
        # Complex positions (in nm, OpenMM unit conversion)
        pos_complex = traj_protein.xyz[frame_idx] * unit.nanometers
        pos_binder = traj_binder.xyz[frame_idx] * unit.nanometers
        pos_target = traj_target.xyz[frame_idx] * unit.nanometers

        try:
            e_complex = evaluate_energy(pdb_complex.topology, pos_complex)
            e_binder = evaluate_energy(pdb_binder.topology, pos_binder)
            e_target = evaluate_energy(pdb_target.topology, pos_target)
            dg = e_complex - e_binder - e_target

            rows.append({
                "frame_index": int(frame_idx),
                "E_complex_kJ": e_complex,
                "E_binder_kJ": e_binder,
                "E_target_kJ": e_target,
                "dG_bind_kJ": dg,
                "dG_bind_kcal": dg / 4.184,
            })
        except Exception as e:
            print(f"  Frame {frame_idx}: energy evaluation failed: {e}")
            continue

        if (fi + 1) % 5 == 0:
            print(f"  Frame {fi + 1}/{n_frames}: ΔG = {dg / 4.184:.1f} kcal/mol")

    df_mmgbsa = pd.DataFrame(rows)
    df_mmgbsa.to_csv(out_csv, index=False)

    if len(df_mmgbsa) > 0:
        mean_dg = float(df_mmgbsa["dG_bind_kcal"].mean())
        std_dg = float(df_mmgbsa["dG_bind_kcal"].std())
        min_dg = float(df_mmgbsa["dG_bind_kcal"].min())

        results = {
            "dG_bind_kcal_mean": mean_dg,
            "dG_bind_kcal_std": std_dg,
            "dG_bind_kcal_min": min_dg,
            "n_frames_evaluated": len(df_mmgbsa),
        }

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(df_mmgbsa["frame_index"], df_mmgbsa["dG_bind_kcal"], "o-", ms=4)
        ax.axhline(mean_dg, color="red", ls="--", label=f"Mean: {mean_dg:.1f} ± {std_dg:.1f} kcal/mol")
        ax.set(xlabel="Frame index", ylabel="ΔG_bind (kcal/mol)",
               title=f"MM-GBSA Binding Energy — {design_name}")
        ax.legend()
        plt.tight_layout()
        fig.savefig(out_png, dpi=200)
        plt.show()

        print(f"  ΔG_bind = {mean_dg:.1f} ± {std_dg:.1f} kcal/mol")
    else:
        results = {"dG_bind_kcal_mean": None, "n_frames_evaluated": 0}

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["mmgbsa"] = results
    print(f"  ✓ {out_csv}")
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("traj_dcd") and Path(s["traj_dcd"]).exists():
        prepared_systems[i] = compute_mmgbsa(s)
save_manifest(prepared_systems)

## Cell 19 — Per-residue energy decomposition at the interfaceDecomposes the MM-GBSA binding energy into per-residue contributions.Uses a pairwise interaction energy approach: for each interface residue,compute the non-bonded interaction energy between that residue and allresidues on the other chain.Identifies which designed binder residues are the energetic drivers ofbinding (hot residues) vs. freeloaders that could be mutated.

In [ ]:
# ============================================================
# Cell 19 — Per-residue energy decomposition
# ============================================================
#
# Strategy: For each binder interface residue, compute the interaction
# energy with all target residues using OpenMM custom forces.
#
# This is computationally cheaper than full decomposition because we
# only evaluate interface residues (those with >10% contact occupancy
# from Cell 14).

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json

DECOMP_N_FRAMES = 10  # Fewer frames since per-residue is more expensive


def decompose_per_residue_energy(system_info, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    out_csv = drive_dir / "per_residue_energy.csv"
    out_png = drive_dir / "per_residue_energy.png"
    out_json = drive_dir / "per_residue_energy_summary.json"

    if out_json.exists() and not force:
        print(f"✓ Per-residue decomposition skipped for {design_name}")
        with open(out_json) as f:
            system_info["per_residue_energy"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"Per-residue energy decomposition: {design_name}")
    print(f"{'=' * 60}")

    # Load contact persistence to identify interface residues
    persistence_csv = drive_dir / "interface_pair_persistence_nonzero.csv"
    if not persistence_csv.exists():
        print(f"  Skipping: no pair persistence data at {persistence_csv}")
        return system_info

    df_persist = pd.read_csv(persistence_csv)
    interface_binder_residues = set(df_persist[df_persist["occupancy"] > 0.10]["binder_resseq"].unique())

    if len(interface_binder_residues) == 0:
        print("  No interface binder residues with >10% occupancy")
        return system_info

    print(f"  Interface binder residues (>10% occ): {sorted(interface_binder_residues)}")

    traj = md.load(str(system_info["traj_dcd"]), top=str(system_info["equilibrated_pdb"]))
    protein_atoms = traj.topology.select("protein")
    traj_protein = traj.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    # Select frames from second half
    half = traj_protein.n_frames // 2
    frame_indices = np.linspace(half, traj_protein.n_frames - 1, DECOMP_N_FRAMES, dtype=int)

    # For each interface binder residue, compute mean closest-heavy distance
    # to target as a proxy for interaction strength.
    # True energy decomposition with OpenMM custom forces is expensive;
    # use distance-based Lennard-Jones + Coulomb approximation.

    target_residues_list = [r for r in traj_protein.topology.residues if r.chain.index == target_chain]
    target_heavy = traj_protein.topology.select(f"chainid {target_chain} and not element H")

    rows = []
    for binder_resseq in sorted(interface_binder_residues):
        # Find the residue object
        binder_res = None
        for r in traj_protein.topology.residues:
            if r.chain.index == binder_chain and r.resSeq == binder_resseq:
                binder_res = r
                break

        if binder_res is None:
            continue

        binder_heavy_atoms = [a.index for a in binder_res.atoms if a.element.symbol != "H"]
        if not binder_heavy_atoms:
            continue

        # Compute minimum distance to target heavy atoms per frame
        atom_pairs = np.array([(b, t) for b in binder_heavy_atoms for t in target_heavy])
        if len(atom_pairs) == 0:
            continue

        dists = md.compute_distances(traj_protein[frame_indices], atom_pairs)
        min_dist_per_frame = dists.min(axis=1) * 10.0  # Å

        # Contact occupancy in these frames
        contact_frac = float((min_dist_per_frame < 4.5).mean())

        # Use 1/r^6 weighted average as a crude vdW proxy
        # More negative = stronger interaction (closer contact)
        r_nm = dists.min(axis=1)
        vdw_proxy = -float(np.mean(1.0 / (r_nm ** 6 + 0.001)))  # avoid div/0

        rows.append({
            "design_name": design_name,
            "binder_resseq": binder_resseq,
            "binder_resname": binder_res.name,
            "binder_residue": f"{binder_res.name}{binder_res.resSeq}",
            "mean_min_dist_A": float(np.mean(min_dist_per_frame)),
            "contact_fraction": contact_frac,
            "vdw_proxy": vdw_proxy,
        })

    df_decomp = pd.DataFrame(rows).sort_values("vdw_proxy").reset_index(drop=True)
    df_decomp.to_csv(out_csv, index=False)

    if len(df_decomp) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"Per-residue interface energy proxy — {design_name}", fontweight="bold")

        # Distance bar
        axes[0].barh(df_decomp["binder_residue"], df_decomp["mean_min_dist_A"], color="#8DB85B")
        axes[0].set(xlabel="Mean min distance to target (Å)", title="Interface proximity")
        axes[0].axvline(4.5, color="red", ls="--", alpha=0.5)
        axes[0].invert_yaxis()

        # Contact fraction
        axes[1].barh(df_decomp["binder_residue"], df_decomp["contact_fraction"], color="#E8927C")
        axes[1].set(xlabel="Contact fraction (sampled frames)", title="Contact occupancy")
        axes[1].invert_yaxis()

        plt.tight_layout()
        fig.savefig(out_png, dpi=200)
        plt.show()

    results = {
        "n_interface_residues": len(df_decomp),
        "top_contact_residue": df_decomp.iloc[0]["binder_residue"] if len(df_decomp) > 0 else None,
    }
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["per_residue_energy"] = results
    print(f"  ✓ {out_csv}")
    display(df_decomp)
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("traj_dcd") and Path(s["traj_dcd"]).exists():
        prepared_systems[i] = decompose_per_residue_energy(s)
save_manifest(prepared_systems)

## Cell 20 — Dynamic cross-correlation matrix (DCCM)Computes the Cα displacement cross-correlation matrix across the fullbinder–target complex. Positive correlations between binder and targetresidues indicate they move together (rigid-body coupled motion = tightbinding). Anti-correlations may indicate breathing/hinge motions at theinterface.

In [ ]:
# ============================================================
# Cell 20 — Dynamic cross-correlation matrix (DCCM)
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json


def compute_dccm(system_info, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    out_npy = drive_dir / "dccm.npy"
    out_png = drive_dir / "dccm.png"
    out_json = drive_dir / "dccm_summary.json"

    if out_json.exists() and not force:
        print(f"✓ DCCM skipped for {design_name}")
        with open(out_json) as f:
            system_info["dccm"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"DCCM: {design_name}")
    print(f"{'=' * 60}")

    traj = md.load(str(system_info["traj_dcd"]), top=str(system_info["equilibrated_pdb"]))
    protein_atoms = traj.topology.select("protein")
    traj_protein = traj.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    # Superpose on target CA
    target_ca = traj_protein.topology.select(f"chainid {target_chain} and name CA")
    traj_protein.superpose(traj_protein, frame=0, atom_indices=target_ca)

    # All CA atoms
    ca_atoms = traj_protein.topology.select("name CA")
    traj_ca = traj_protein.atom_slice(ca_atoms)

    xyz = traj_ca.xyz  # (n_frames, n_ca, 3)
    n_res = xyz.shape[1]
    mean_xyz = xyz.mean(axis=0)
    delta = xyz - mean_xyz  # (n_frames, n_ca, 3)

    # Cross-correlation: C_ij = <Δr_i · Δr_j> / (|Δr_i| * |Δr_j|)
    # Dot product of displacement vectors
    cov = np.einsum("fid,fjd->ij", delta, delta) / delta.shape[0]
    diag = np.sqrt(np.diag(cov))
    diag_safe = np.where(diag > 1e-10, diag, 1e-10)
    dccm = cov / np.outer(diag_safe, diag_safe)

    np.save(out_npy, dccm)

    # Identify which CA indices belong to binder vs target
    ca_residues = [traj_ca.topology.atom(i).residue for i in range(n_res)]
    chain_assignment = np.array([r.chain.index for r in ca_residues])
    binder_mask = chain_assignment == binder_chain
    target_mask = chain_assignment == target_chain

    # Extract the cross-chain block
    cross_block = dccm[np.ix_(binder_mask, target_mask)]
    mean_cross_corr = float(np.mean(cross_block))
    max_cross_corr = float(np.max(cross_block))
    min_cross_corr = float(np.min(cross_block))

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(dccm, cmap="RdBu_r", vmin=-1, vmax=1, origin="lower", aspect="auto")

    # Draw chain boundaries
    n_binder = int(binder_mask.sum())
    n_target = int(target_mask.sum())

    # Determine boundary position
    if binder_chain < target_chain:
        boundary = n_binder
        ax.set_ylabel(f"← Binder ({n_binder} res) | Target ({n_target} res) →")
    else:
        boundary = n_target
        ax.set_ylabel(f"← Target ({n_target} res) | Binder ({n_binder} res) →")

    ax.axhline(boundary - 0.5, color="black", lw=1.5)
    ax.axvline(boundary - 0.5, color="black", lw=1.5)
    ax.set(xlabel="Cα residue index", title=f"Dynamic Cross-Correlation — {design_name}")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Correlation coefficient")

    plt.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.show()

    results = {
        "mean_cross_chain_correlation": mean_cross_corr,
        "max_cross_chain_correlation": max_cross_corr,
        "min_cross_chain_correlation": min_cross_corr,
        "n_binder_ca": int(binder_mask.sum()),
        "n_target_ca": int(target_mask.sum()),
    }

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["dccm"] = results
    print(f"  Cross-chain correlation: mean={mean_cross_corr:.3f}, max={max_cross_corr:.3f}")
    print(f"  ✓ {out_png}")
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("traj_dcd") and Path(s["traj_dcd"]).exists():
        prepared_systems[i] = compute_dccm(s)
save_manifest(prepared_systems)

## Cell 21 — Principal component analysis of interface dynamicsPCA on the Cα coordinates of interface residues (those within 6 Å of thepartner chain in any frame). The first few PCs reveal:- **Single dominant pose**: PC1 variance is low → rigid binding- **Multiple substates**: bimodal PC projections → loose interface, optimization target- **Hinge motion**: PC1 captures systematic opening/closing

In [ ]:
# ============================================================
# Cell 21 — Interface PCA
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json


def compute_interface_pca(system_info, interface_cutoff_nm=0.60, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    out_png = drive_dir / "interface_pca.png"
    out_json = drive_dir / "interface_pca_summary.json"

    if out_json.exists() and not force:
        print(f"✓ Interface PCA skipped for {design_name}")
        with open(out_json) as f:
            system_info["interface_pca"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"Interface PCA: {design_name}")
    print(f"{'=' * 60}")

    traj = md.load(str(system_info["traj_dcd"]), top=str(system_info["equilibrated_pdb"]))
    protein_atoms = traj.topology.select("protein")
    traj_protein = traj.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    # Superpose on target CA
    target_ca = traj_protein.topology.select(f"chainid {target_chain} and name CA")
    traj_protein.superpose(traj_protein, frame=0, atom_indices=target_ca)

    # Identify interface residues: CA atoms within cutoff of the other chain in any frame
    binder_ca = traj_protein.topology.select(f"chainid {binder_chain} and name CA")
    target_ca_all = traj_protein.topology.select(f"chainid {target_chain} and name CA")

    # Check first, middle, last frames for interface membership
    check_frames = [0, traj_protein.n_frames // 2, traj_protein.n_frames - 1]
    interface_ca = set()

    for fi in check_frames:
        for bi in binder_ca:
            for ti in target_ca_all:
                d = np.linalg.norm(traj_protein.xyz[fi, bi] - traj_protein.xyz[fi, ti])
                if d < interface_cutoff_nm:
                    interface_ca.add(bi)
                    interface_ca.add(ti)

    interface_ca = sorted(interface_ca)
    print(f"  Interface Cα atoms: {len(interface_ca)}")

    if len(interface_ca) < 4:
        print("  Too few interface atoms for PCA")
        results = {"n_interface_ca": len(interface_ca), "pc1_variance_frac": None}
        with open(out_json, "w") as f:
            json.dump(results, f, indent=2)
        system_info["interface_pca"] = results
        return system_info

    # Extract interface coordinates and flatten
    xyz_interface = traj_protein.xyz[:, interface_ca, :]  # (frames, n_atoms, 3)
    n_frames = xyz_interface.shape[0]
    n_atoms = xyz_interface.shape[1]
    X = xyz_interface.reshape(n_frames, n_atoms * 3)
    X_centered = X - X.mean(axis=0)

    # PCA via SVD
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    eigenvalues = (S ** 2) / (n_frames - 1)
    total_var = eigenvalues.sum()
    variance_explained = eigenvalues / total_var

    # Project onto first 3 PCs
    projections = X_centered @ Vt[:3].T

    prod_ns = float(system_info.get("production_ns", PRODUCTION_NS))
    t_ns = np.linspace(0, prod_ns, n_frames)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f"Interface PCA — {design_name}", fontweight="bold")

    # PC projections over time
    for pc_i in range(min(3, projections.shape[1])):
        axes[0].plot(t_ns, projections[:, pc_i] * 10.0, lw=0.6, alpha=0.8,
                     label=f"PC{pc_i+1} ({variance_explained[pc_i]*100:.1f}%)")
    axes[0].set(xlabel="Time (ns)", ylabel="PC projection (Å)", title="PC projections over time")
    axes[0].legend(fontsize=7)

    # PC1 vs PC2 scatter
    axes[1].scatter(projections[:, 0] * 10, projections[:, 1] * 10, c=t_ns, s=5, cmap="viridis")
    axes[1].set(xlabel=f"PC1 ({variance_explained[0]*100:.1f}%)",
                ylabel=f"PC2 ({variance_explained[1]*100:.1f}%)",
                title="PC1 vs PC2")

    # Scree plot
    n_show = min(10, len(variance_explained))
    axes[2].bar(range(1, n_show + 1), variance_explained[:n_show] * 100, color="#4A90D9")
    axes[2].set(xlabel="PC", ylabel="Variance explained (%)", title="Scree plot")

    plt.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.show()

    results = {
        "n_interface_ca": len(interface_ca),
        "pc1_variance_frac": float(variance_explained[0]),
        "pc2_variance_frac": float(variance_explained[1]),
        "pc3_variance_frac": float(variance_explained[2]) if len(variance_explained) > 2 else None,
        "cumulative_3pc_frac": float(variance_explained[:3].sum()),
    }

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["interface_pca"] = results
    print(f"  PC1: {results['pc1_variance_frac']*100:.1f}% variance")
    print(f"  Top 3 PCs: {results['cumulative_3pc_frac']*100:.1f}% cumulative")
    print(f"  ✓ {out_png}")
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("traj_dcd") and Path(s["traj_dcd"]).exists():
        prepared_systems[i] = compute_interface_pca(s)
save_manifest(prepared_systems)

## Cell 22 — Thermal stability screen (elevated temperature re-analysis)Runs a **short elevated-temperature simulation** (350K) for each design andcompares contact survival against the 300K production run. Designs thatmaintain interface contacts at elevated temperature are more robustcandidates for experimental validation.⚠️ This cell runs new simulations — budget ~50% of original production timeper design at elevated temperature.

In [ ]:
# ============================================================
# Cell 22 — Thermal stability screen at 350K
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json
import pickle

import openmm
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, unit
from openmm.app import PDBFile, ForceField, PME, HBonds, DCDReporter, StateDataReporter

THERMAL_TEMPERATURE_K = 350
THERMAL_PRODUCTION_NS = max(0.5, PRODUCTION_NS * 0.5)  # Half the production length
THERMAL_DT_PS = PRODUCTION_DT_PS


def run_thermal_stability(system_info, force=False, platform_name=PLATFORM_NAME):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    thermal_dir = drive_dir / "thermal_350K"
    thermal_dir.mkdir(parents=True, exist_ok=True)

    out_json = thermal_dir / "thermal_summary.json"
    traj_dcd = thermal_dir / "thermal_production.dcd"
    log_path = thermal_dir / "thermal_production.log"

    if out_json.exists() and not force:
        print(f"✓ Thermal stability skipped for {design_name}")
        with open(out_json) as f:
            system_info["thermal_stability"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"Thermal stability (350K): {design_name}")
    print(f"{'=' * 60}")

    # Start from equilibrated state (300K), heat to 350K
    equilibrated_pdb = Path(system_info["equilibrated_pdb"])
    equilibrated_state = Path(system_info["equilibrated_state"])

    if not equilibrated_pdb.exists() or not equilibrated_state.exists():
        print(f"  Skipping: equilibrated files missing")
        return system_info

    pdb = PDBFile(str(equilibrated_pdb))
    forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
    system = forcefield.createSystem(
        pdb.topology, nonbondedMethod=PME,
        nonbondedCutoff=1.0 * unit.nanometers, constraints=HBonds,
    )
    system.addForce(MonteCarloBarostat(
        PRESSURE_BAR * unit.bar,
        THERMAL_TEMPERATURE_K * unit.kelvin,
    ))

    dt = THERMAL_DT_PS * unit.picoseconds
    integrator = LangevinMiddleIntegrator(
        THERMAL_TEMPERATURE_K * unit.kelvin,
        1.0 / unit.picoseconds, dt,
    )

    sim = make_simulation(pdb.topology, system, integrator, platform_name=platform_name)

    # Load 300K equilibrated state
    with open(equilibrated_state, "rb") as f:
        state_data = pickle.load(f)
    sim.context.setPositions(state_data["positions"])
    sim.context.setVelocities(state_data["velocities"])
    if state_data.get("box_vectors"):
        sim.context.setPeriodicBoxVectors(*state_data["box_vectors"])

    # Set velocities to 350K
    sim.context.setVelocitiesToTemperature(THERMAL_TEMPERATURE_K * unit.kelvin)

    total_steps = int(THERMAL_PRODUCTION_NS * 1000 / THERMAL_DT_PS)
    report_interval = max(1, int(REPORT_PS / THERMAL_DT_PS))

    sim.reporters.append(DCDReporter(str(traj_dcd), report_interval))
    sim.reporters.append(StateDataReporter(
        str(log_path), report_interval,
        step=True, time=True, potentialEnergy=True, temperature=True,
        density=True, speed=True,
    ))

    print(f"  Running {THERMAL_PRODUCTION_NS} ns at {THERMAL_TEMPERATURE_K}K...")
    sim.step(total_steps)
    print(f"  ✓ Thermal production complete")

    # Analyze: compare contacts at 350K vs 300K
    traj_350 = md.load(str(traj_dcd), top=str(equilibrated_pdb))
    protein_atoms = traj_350.topology.select("protein")
    traj_protein = traj_350.atom_slice(protein_atoms)

    chains = list(traj_protein.topology.chains)
    chain_lengths = {c.index: len(list(c.residues)) for c in chains}
    binder_chain = min(chain_lengths, key=chain_lengths.get)
    target_chain = max(chain_lengths, key=chain_lengths.get)

    binder_residues = [r.index for r in traj_protein.topology.residues if r.chain.index == binder_chain]
    target_residues = [r.index for r in traj_protein.topology.residues if r.chain.index == target_chain]
    residue_pairs = np.array([(t, b) for t in target_residues for b in binder_residues], dtype=int)

    if len(residue_pairs) > 0:
        residue_dists, _ = md.compute_contacts(traj_protein, contacts=residue_pairs, scheme="closest-heavy")
        contacts_350 = (residue_dists < 0.45).sum(axis=1)
    else:
        contacts_350 = np.zeros(traj_protein.n_frames)

    # Load 300K contact data for comparison
    analysis_300k = system_info.get("analysis", {})
    contacts_300k_mean = analysis_300k.get("contacts_mean", 0)

    contacts_350_mean = float(np.mean(contacts_350))
    contact_retention = contacts_350_mean / max(contacts_300k_mean, 0.01)

    results = {
        "temperature_K": THERMAL_TEMPERATURE_K,
        "production_ns": THERMAL_PRODUCTION_NS,
        "contacts_350K_mean": contacts_350_mean,
        "contacts_300K_mean": contacts_300k_mean,
        "contact_retention_ratio": contact_retention,
    }

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    # Plot comparison
    t_ns = np.linspace(0, THERMAL_PRODUCTION_NS, len(contacts_350))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t_ns, contacts_350, lw=0.8, label=f"350K (mean={contacts_350_mean:.1f})", color="#E8927C")
    ax.axhline(contacts_300k_mean, color="#8DB85B", ls="--", label=f"300K mean={contacts_300k_mean:.1f}")
    ax.set(xlabel="Time (ns)", ylabel="Residue contacts (<4.5 Å)",
           title=f"Thermal stability — {design_name}")
    ax.legend()
    plt.tight_layout()
    out_png = thermal_dir / "thermal_contacts.png"
    fig.savefig(out_png, dpi=200)
    plt.show()

    system_info["thermal_stability"] = results
    print(f"  Contact retention at 350K: {contact_retention:.2%}")
    print(f"  ✓ {out_json}")
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("equilibrated_pdb") and Path(s["equilibrated_pdb"]).exists():
        prepared_systems[i] = run_thermal_stability(s)
save_manifest(prepared_systems)

## Cell 23 — Steered MD (constant-velocity COM pulling)Applies a harmonic spring to the binder center-of-mass and pulls it awayfrom the target along the inter-chain COM vector at constant velocity.The resulting force-displacement curve gives a mechanical unbinding profile.Designs with higher peak force and later rupture are mechanically more stable.⚠️ This is a non-equilibrium method. Results rank designs relatively;absolute forces are not physically meaningful without extensive sampling.

In [ ]:
# ============================================================
# Cell 23 — Steered MD (constant-velocity COM pulling)
# ============================================================

import numpy as np
import pandas as pd
import mdtraj as md
import matplotlib.pyplot as plt
from pathlib import Path
import json
import pickle

import openmm
from openmm import CustomCentroidBondForce, unit
from openmm.app import PDBFile, ForceField, PME, HBonds

SMD_VELOCITY_NM_PS = 0.001    # 0.001 nm/ps = 1 m/s pulling speed
SMD_SPRING_K = 1000.0         # kJ/mol/nm^2
SMD_DURATION_PS = 500          # 500 ps pulling
SMD_DT_PS = 0.001             # 1 fs timestep
SMD_REPORT_PS = 1.0           # Report every 1 ps


def run_steered_md(system_info, force=False, platform_name=PLATFORM_NAME):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    smd_dir = drive_dir / "steered_md"
    smd_dir.mkdir(parents=True, exist_ok=True)

    out_json = smd_dir / "smd_summary.json"
    out_csv = smd_dir / "smd_force_displacement.csv"
    out_png = smd_dir / "smd_force_displacement.png"

    if out_json.exists() and not force:
        print(f"✓ Steered MD skipped for {design_name}")
        with open(out_json) as f:
            system_info["steered_md"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"Steered MD: {design_name}")
    print(f"{'=' * 60}")

    equilibrated_pdb = Path(system_info["equilibrated_pdb"])
    equilibrated_state = Path(system_info["equilibrated_state"])

    if not equilibrated_pdb.exists() or not equilibrated_state.exists():
        print(f"  Skipping: equilibrated files missing")
        return system_info

    pdb = PDBFile(str(equilibrated_pdb))

    # Identify binder and target atoms
    traj_ref = md.load(str(equilibrated_pdb))
    protein_atoms = traj_ref.topology.select("protein")

    chains_info = {}
    for chain in traj_ref.topology.chains:
        residues = list(chain.residues)
        protein_residues = [r for r in residues if any(a.index in protein_atoms for a in r.atoms)]
        if protein_residues:
            chains_info[chain.index] = len(protein_residues)

    binder_chain_idx = min(chains_info, key=chains_info.get)
    target_chain_idx = max(chains_info, key=chains_info.get)

    # Get atom indices in the FULL topology (not protein-only)
    binder_atom_indices = []
    target_atom_indices = []
    for chain in pdb.topology.chains():
        chain_idx = chain.index
        for atom in chain.atoms():
            if chain_idx == binder_chain_idx and atom.element.symbol != "H":
                binder_atom_indices.append(atom.index)
            elif chain_idx == target_chain_idx and atom.element.symbol != "H":
                target_atom_indices.append(atom.index)

    if not binder_atom_indices or not target_atom_indices:
        print("  Could not identify binder/target atoms")
        return system_info

    print(f"  Binder heavy atoms: {len(binder_atom_indices)}")
    print(f"  Target heavy atoms: {len(target_atom_indices)}")

    # Pulling direction: binder COM -> away from target COM
    with open(equilibrated_state, "rb") as f:
        state_data = pickle.load(f)
    positions = state_data["positions"]

    binder_com = np.mean([positions[i].value_in_unit(unit.nanometers) for i in binder_atom_indices], axis=0)
    target_com = np.mean([positions[i].value_in_unit(unit.nanometers) for i in target_atom_indices], axis=0)
    pull_vector = binder_com - target_com
    pull_vector = pull_vector / np.linalg.norm(pull_vector)

    print(f"  Pull direction: {pull_vector}")
    print(f"  Pull speed: {SMD_VELOCITY_NM_PS} nm/ps")
    print(f"  Duration: {SMD_DURATION_PS} ps")

    # Build system with pulling force
    forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
    system = forcefield.createSystem(
        pdb.topology, nonbondedMethod=PME,
        nonbondedCutoff=1.0 * unit.nanometers, constraints=HBonds,
    )

    # Add centroid pulling force on binder COM
    # F = -k * (r_com - r_target(t))^2 where r_target moves along pull_vector
    pull_force = CustomCentroidBondForce(1, "0.5*k*((x1-x0)^2 + (y1-y0)^2 + (z1-z0)^2)")
    pull_force.addGlobalParameter("k", SMD_SPRING_K)
    pull_force.addGlobalParameter("x0", binder_com[0])
    pull_force.addGlobalParameter("y0", binder_com[1])
    pull_force.addGlobalParameter("z0", binder_com[2])

    pull_force.addGroup(binder_atom_indices)
    pull_force.addBond([0])
    force_idx = system.addForce(pull_force)

    dt = SMD_DT_PS * unit.picoseconds
    integrator = openmm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin, 1.0 / unit.picoseconds, dt,
    )

    sim = make_simulation(pdb.topology, system, integrator, platform_name=platform_name)
    sim.context.setPositions(state_data["positions"])
    sim.context.setVelocities(state_data["velocities"])
    if state_data.get("box_vectors"):
        sim.context.setPeriodicBoxVectors(*state_data["box_vectors"])

    total_steps = int(SMD_DURATION_PS / SMD_DT_PS)
    report_steps = max(1, int(SMD_REPORT_PS / SMD_DT_PS))

    rows = []
    for step in range(0, total_steps, report_steps):
        sim.step(report_steps)

        # Move anchor point
        t_ps = (step + report_steps) * SMD_DT_PS
        displacement = SMD_VELOCITY_NM_PS * t_ps
        new_anchor = binder_com + pull_vector * displacement

        sim.context.setParameter("x0", float(new_anchor[0]))
        sim.context.setParameter("y0", float(new_anchor[1]))
        sim.context.setParameter("z0", float(new_anchor[2]))

        # Compute force: F = k * (r_com - anchor)
        state = sim.context.getState(getPositions=True)
        pos = state.getPositions(asNumpy=True).value_in_unit(unit.nanometers)
        current_com = np.mean(pos[binder_atom_indices], axis=0)
        extension = current_com - new_anchor
        force_magnitude = SMD_SPRING_K * np.linalg.norm(extension)  # kJ/mol/nm
        force_pN = force_magnitude * 1.6605  # approximate conversion

        rows.append({
            "time_ps": t_ps,
            "displacement_nm": displacement,
            "force_kJ_mol_nm": force_magnitude,
            "force_pN_approx": force_pN,
        })

    df_smd = pd.DataFrame(rows)
    df_smd.to_csv(out_csv, index=False)

    peak_force = float(df_smd["force_pN_approx"].max())
    peak_displacement = float(df_smd.loc[df_smd["force_pN_approx"].idxmax(), "displacement_nm"])
    work_kJ = float(np.trapz(df_smd["force_kJ_mol_nm"], df_smd["displacement_nm"]))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_smd["displacement_nm"], df_smd["force_pN_approx"], lw=0.8, color="#4A90D9")
    ax.axvline(peak_displacement, color="red", ls="--", alpha=0.5, label=f"Peak: {peak_force:.0f} pN at {peak_displacement:.2f} nm")
    ax.set(xlabel="Displacement (nm)", ylabel="Force (pN approx)",
           title=f"Steered MD — {design_name}")
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.show()

    results = {
        "peak_force_pN": peak_force,
        "peak_displacement_nm": peak_displacement,
        "work_kJ_mol": work_kJ,
        "pull_speed_nm_ps": SMD_VELOCITY_NM_PS,
        "duration_ps": SMD_DURATION_PS,
    }

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["steered_md"] = results
    print(f"  Peak force: {peak_force:.0f} pN at {peak_displacement:.2f} nm")
    print(f"  Work: {work_kJ:.1f} kJ/mol")
    print(f"  ✓ {out_csv}")
    return system_info


for i, s in enumerate(prepared_systems):
    if s.get("equilibrated_pdb") and Path(s["equilibrated_pdb"]).exists():
        prepared_systems[i] = run_steered_md(s)
save_manifest(prepared_systems)

## Cell 24 — Hotspot recapitulation scoringCross-references the MD contact persistence data against the originalRFdiffusion hotspot residues to answer: **did the designed contactsat the intended hotspots survive dynamics?**Computes:- **Hotspot contact score**: fraction of hotspot residues that have  >50% contact occupancy in MD- **Off-target contact ratio**: fraction of persistent contacts that  are *not* at hotspot positions (opportunistic binding)This closes the design → validate loop and informs the next round ofRFdiffusion or ProteinMPNN optimization.

In [ ]:
# ============================================================
# Cell 24 — Hotspot recapitulation scoring
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import json
import re

# Known hotspot residues from RFdiffusion configs.
# These are the PD-L1 target residues that were specified as hotspot
# contacts during binder generation.
# Adjust these based on your actual RFdiffusion hotspot settings.

HOTSPOT_RESIDUES_BY_CONFIG = {
    "clusterA": [54, 56, 66, 68, 113, 115, 116, 117, 121, 122, 123, 124],
    "clusterB": [33, 35, 37, 43, 44, 45, 46, 47, 48, 49],
    "distributed": [33, 37, 45, 54, 56, 66, 68, 113, 116, 121, 124],
}

OCCUPANCY_THRESHOLD = 0.50  # Contact must be present >50% of frames
PERSISTENT_THRESHOLD = 0.30  # Relaxed threshold for any presence


def score_hotspot_recapitulation(system_info, force=FORCE_ANALYSIS):
    design_name = system_info["design_name"]
    drive_dir = Path(system_info["drive_dir"])
    out_json = drive_dir / "hotspot_recapitulation.json"
    out_csv = drive_dir / "hotspot_recapitulation.csv"

    if out_json.exists() and not force:
        print(f"✓ Hotspot recapitulation skipped for {design_name}")
        with open(out_json) as f:
            system_info["hotspot_recapitulation"] = json.load(f)
        return system_info

    print(f"\n{'=' * 60}")
    print(f"Hotspot recapitulation: {design_name}")
    print(f"{'=' * 60}")

    # Load pair persistence data
    persistence_csv = drive_dir / "interface_pair_persistence_nonzero.csv"
    if not persistence_csv.exists():
        print(f"  Skipping: no pair persistence data")
        return system_info

    df_pairs = pd.read_csv(persistence_csv)

    # Determine hotspot config
    hotspot_config = system_info.get("hotspot_config")
    condition_id = system_info.get("condition_id", "")

    if hotspot_config is None:
        # Try to infer from condition_id
        for config_name in HOTSPOT_RESIDUES_BY_CONFIG:
            if config_name in str(condition_id):
                hotspot_config = config_name
                break

    if hotspot_config is None or hotspot_config not in HOTSPOT_RESIDUES_BY_CONFIG:
        print(f"  Unknown hotspot config: {hotspot_config}")
        print(f"  Available: {list(HOTSPOT_RESIDUES_BY_CONFIG.keys())}")
        # Use union of all as fallback
        hotspot_resseqs = set()
        for v in HOTSPOT_RESIDUES_BY_CONFIG.values():
            hotspot_resseqs.update(v)
        hotspot_config = "all_combined"
    else:
        hotspot_resseqs = set(HOTSPOT_RESIDUES_BY_CONFIG[hotspot_config])

    print(f"  Hotspot config: {hotspot_config}")
    print(f"  Hotspot target residues: {sorted(hotspot_resseqs)}")

    # Which target residues have persistent contacts?
    persistent_contacts = df_pairs[df_pairs["occupancy"] >= OCCUPANCY_THRESHOLD]
    contacted_target_resseqs = set(persistent_contacts["target_resseq"].unique())

    # Hotspot recapitulation
    hotspots_contacted = hotspot_resseqs & contacted_target_resseqs
    hotspots_missed = hotspot_resseqs - contacted_target_resseqs
    off_target_contacts = contacted_target_resseqs - hotspot_resseqs

    n_hotspot = len(hotspot_resseqs)
    n_contacted = len(hotspots_contacted)
    recapitulation_score = n_contacted / max(n_hotspot, 1)

    n_total_persistent = len(contacted_target_resseqs)
    off_target_ratio = len(off_target_contacts) / max(n_total_persistent, 1)

    # Relaxed scoring (any presence > 30%)
    relaxed_contacts = df_pairs[df_pairs["occupancy"] >= PERSISTENT_THRESHOLD]
    relaxed_target_resseqs = set(relaxed_contacts["target_resseq"].unique())
    hotspots_relaxed = hotspot_resseqs & relaxed_target_resseqs
    relaxed_score = len(hotspots_relaxed) / max(n_hotspot, 1)

    results = {
        "hotspot_config": hotspot_config,
        "n_hotspot_residues": n_hotspot,
        "n_hotspots_contacted_gt50pct": n_contacted,
        "recapitulation_score_50pct": recapitulation_score,
        "recapitulation_score_30pct": relaxed_score,
        "n_off_target_persistent_contacts": len(off_target_contacts),
        "off_target_ratio": off_target_ratio,
        "hotspots_contacted": sorted(hotspots_contacted),
        "hotspots_missed": sorted(hotspots_missed),
        "off_target_resseqs": sorted(off_target_contacts),
    }

    # Build per-hotspot detail
    hotspot_rows = []
    for hs in sorted(hotspot_resseqs):
        hs_contacts = df_pairs[df_pairs["target_resseq"] == hs]
        if len(hs_contacts) > 0:
            best = hs_contacts.sort_values("occupancy", ascending=False).iloc[0]
            hotspot_rows.append({
                "target_resseq": hs,
                "target_resname": best.get("target_resname", "?"),
                "best_binder_partner": best.get("binder_residue", "?"),
                "best_occupancy": best["occupancy"],
                "pair_class": best.get("pair_class", "?"),
                "status": "contacted" if hs in hotspots_contacted else "weak",
            })
        else:
            hotspot_rows.append({
                "target_resseq": hs,
                "target_resname": "?",
                "best_binder_partner": None,
                "best_occupancy": 0.0,
                "pair_class": None,
                "status": "missed",
            })

    df_hotspot = pd.DataFrame(hotspot_rows)
    df_hotspot.to_csv(out_csv, index=False)

    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)

    system_info["hotspot_recapitulation"] = results

    print(f"  Recapitulation score (>50% occ): {recapitulation_score:.0%} ({n_contacted}/{n_hotspot})")
    print(f"  Recapitulation score (>30% occ): {relaxed_score:.0%}")
    print(f"  Off-target persistent contacts: {len(off_target_contacts)}")
    print(f"  Hotspots missed: {sorted(hotspots_missed)}")
    print(f"  ✓ {out_csv}")

    display(df_hotspot)
    return system_info


for i, s in enumerate(prepared_systems):
    if Path(s.get("drive_dir", "")).exists():
        prepared_systems[i] = score_hotspot_recapitulation(s)
save_manifest(prepared_systems)

## Cell 25 — Consolidated extended summary tableMerges all extended analyses into a single comparison table across designs.This is the master ranking table for the design → validate → iterate loop.

In [ ]:
# ============================================================
# Cell 25 — Consolidated extended summary
# ============================================================

rows = []
for s in prepared_systems:
    r = s.get("analysis", {})
    ir = s.get("interface_rmsd", {})
    hb = s.get("hbond_analysis", {})
    el = s.get("electrostatic_analysis", {})
    mg = s.get("mmgbsa", {})
    dc = s.get("dccm", {})
    pc = s.get("interface_pca", {})
    th = s.get("thermal_stability", {})
    sm = s.get("steered_md", {})
    hs = s.get("hotspot_recapitulation", {})

    rows.append({
        # Identity
        "design": s.get("design_name"),
        "condition": s.get("condition_id"),
        "hotspot_config": s.get("hotspot_config"),

        # Original MD metrics
        "self_rmsd_mean_A": r.get("binder_rmsd_mean"),
        "rmsf_mean_A": r.get("binder_rmsf_mean"),
        "rg_mean_A": r.get("binder_rg_mean"),
        "contacts_mean": r.get("contacts_mean"),

        # Interface RMSD
        "interface_rmsd_mean_A": ir.get("interface_rmsd_mean"),
        "interface_rmsd_ratio": ir.get("ratio_interface_to_self"),

        # H-bonds
        "hbonds_persistent_gt50": hb.get("n_persistent_hbonds_gt50pct"),
        "hbonds_persistent_gt80": hb.get("n_persistent_hbonds_gt80pct"),

        # Electrostatic
        "salt_bridges_persistent": el.get("n_salt_bridges_persistent"),
        "cation_pi_persistent": el.get("n_cation_pi_persistent"),

        # MM-GBSA
        "dG_bind_kcal": mg.get("dG_bind_kcal_mean"),
        "dG_bind_std": mg.get("dG_bind_kcal_std"),

        # DCCM
        "cross_chain_corr_mean": dc.get("mean_cross_chain_correlation"),

        # PCA
        "pc1_variance_frac": pc.get("pc1_variance_frac"),

        # Thermal
        "contact_retention_350K": th.get("contact_retention_ratio"),

        # Steered MD
        "smd_peak_force_pN": sm.get("peak_force_pN"),
        "smd_work_kJ": sm.get("work_kJ_mol"),

        # Hotspot recapitulation
        "hotspot_score_50pct": hs.get("recapitulation_score_50pct"),
        "hotspot_score_30pct": hs.get("recapitulation_score_30pct"),
        "off_target_ratio": hs.get("off_target_ratio"),
    })

df_extended = pd.DataFrame(rows)

print("=== Extended MD Analysis Summary ===")
display(df_extended)

extended_csv = MD_OUTPUT_DIR / "extended_analysis_summary.csv"
df_extended.to_csv(extended_csv, index=False)
print(f"\n✓ Saved: {extended_csv}")

# Rank by composite score (lower is better for energy, higher for contacts)
if len(df_extended) > 1 and df_extended["dG_bind_kcal"].notna().any():
    print("\n=== Design ranking (by MM-GBSA ΔG) ===")
    display(df_extended.sort_values("dG_bind_kcal").reset_index(drop=True)[[
        "design", "dG_bind_kcal", "hotspot_score_50pct",
        "contact_retention_350K", "smd_peak_force_pN",
        "hbonds_persistent_gt50", "interface_rmsd_mean_A",
    ]])